In [2]:
# Import library yang dibutuhkan
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas6_ETL_Pipeline") \
    .getOrCreate()

In [3]:
# 1. Membaca data Transaksi (CSV)
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)

# 2. Membaca data Produk (JSON Lines)
df_prod = spark.read.json("tugas6_produk.json")

# 3. Membaca data Ulasan (CSV)
df_rev = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Menampilkan jumlah baris dan schema
print(f"Jumlah baris Transaksi: {df_trx.count()}")
df_trx.printSchema()

print(f"Jumlah baris Produk: {df_prod.count()}")
df_prod.printSchema()

print(f"Jumlah baris Ulasan: {df_rev.count()}")
df_rev.printSchema()

Jumlah baris Transaksi: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah baris Produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah baris Ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [4]:
# Join Transaksi dengan Produk (Inner Join menggunakan product_id)
df_join_prod = df_trx.join(df_prod, on="product_id", how="inner")

# Join hasilnya dengan Ulasan (Left Join menggunakan order_id)
df_joined = df_join_prod.join(df_rev, on="order_id", how="left")

# Menambahkan kolom total_pendapatan (unit_terjual x harga)
df_joined = df_joined.withColumn("total_pendapatan", F.col("unit_terjual") * F.col("harga"))

# Tampilkan beberapa baris untuk verifikasi
df_joined.show(5)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

In [5]:
# 1. Tambahkan kolom ada_ulasan (True/False) SEBELUM menangani null
# Logika: Jika kolom rating tidak null, maka True, sebaliknya False.
df_enriched = df_joined.withColumn("ada_ulasan", F.col("rating").isNotNull())

# 2. Isi nilai kosong pada kolom rating dengan angka 0
# Alasan: Transaksi berstatus null pada rating terjadi karena pelanggan memang belum memberikan ulasan, sehingga nilai 0 merepresentasikan ketiadaan ulasan tersebut.
df_final = df_enriched.fillna(0, subset=["rating"])

# Tampilkan hasil perubahan
df_final.select("order_id", "rating", "ada_ulasan").show(10)

+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|     0|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|     0|     false|
|     TX5|     5|      true|
|     TX6|     4|      true|
|     TX7|     0|     false|
|     TX8|     5|      true|
|     TX9|     0|     false|
+--------+------+----------+
only showing top 10 rows



In [10]:
# Tambahkan hdfs://localhost:9000 (atau port HDFS Anda, misal 8020/9000)
output_path = "hdfs://localhost:9000/user/azizi/tugas6/hasil_etl"

df_final.write.mode("overwrite").partitionBy("kategori").parquet(output_path)
print("Data berhasil disimpan ke HDFS.")

[Stage 31:>                                                         (0 + 1) / 1]

Data berhasil disimpan ke HDFS.


In [12]:
!hdfs dfs -ls -R /user/azizi/tugas6/hasil_etl

-rw-r--r--   3 azizi supergroup          0 2026-10-01 12:21 /user/azizi/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - azizi supergroup          0 2026-10-01 12:21 /user/azizi/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 azizi supergroup      16364 2026-10-01 12:21 /user/azizi/tugas6/hasil_etl/kategori=Elektronik/part-00000-676f98e5-56c1-4f36-942b-3c405723df9e.c000.snappy.parquet
drwxr-xr-x   - azizi supergroup          0 2026-10-01 12:21 /user/azizi/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 azizi supergroup      11134 2026-10-01 12:21 /user/azizi/tugas6/hasil_etl/kategori=Fashion/part-00000-676f98e5-56c1-4f36-942b-3c405723df9e.c000.snappy.parquet
drwxr-xr-x   - azizi supergroup          0 2026-10-01 12:21 /user/azizi/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 azizi supergroup      10393 2026-10-01 12:21 /user/azizi/tugas6/hasil_etl/kategori=Kesehatan/part-00000-676f98e5-56c1-4f36-942b-3c405723df9e.c000.snappy.parquet
drwxr-xr-x   - azizi supergroup          0 2026-10-0

In [13]:
# Membaca kembali data yang telah disimpan di HDFS
df_read_back = spark.read.parquet("hdfs://localhost:9000/user/azizi/tugas6/hasil_etl")

# Menampilkan count() sebagai bukti data tersimpan utuh
print(f"Jumlah baris data setelah di-load: {df_read_back.count()}")

Jumlah baris data setelah di-load: 5000


In [14]:
# Menghitung persentase ulasan per kategori
insight_df = df_final.groupBy("kategori") \
    .agg(
        F.count("*").alias("total_transaksi"),
        F.sum(F.when(F.col("ada_ulasan") == True, 1).otherwise(0)).alias("jumlah_ada_ulasan")
    ) \
    .withColumn("persentase_ulasan", (F.col("jumlah_ada_ulasan") / F.col("total_transaksi")) * 100) \
    .orderBy("persentase_ulasan", ascending=True) # Urutkan dari yang terendah

insight_df.show()

+------------+---------------+-----------------+-----------------+
|    kategori|total_transaksi|jumlah_ada_ulasan|persentase_ulasan|
+------------+---------------+-----------------+-----------------+
|     Makanan|            536|              369|68.84328358208955|
|   Kesehatan|            961|              662|68.88657648283039|
|     Fashion|           1035|              726|70.14492753623188|
|Rumah Tangga|            815|              575| 70.5521472392638|
|  Elektronik|           1653|             1168|  70.659407138536|
+------------+---------------+-----------------+-----------------+

